<a href="https://colab.research.google.com/github/coderhouse2025-droid/Proyecto/blob/main/Proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request, urllib.parse
import xml.etree.ElementTree as ET

BASE_URL = "https://biblioteca.senasa.gob.ar"
BROWSE_ENDPOINT = f"{BASE_URL}/items/browse"

def fetch_rss_page(search, page):
    params = {"output": "rss2", "search": search, "page": page}
    url = f"{BROWSE_ENDPOINT}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "agroexport-rag-scraper/0.2"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        return resp.read().decode("utf-8")

def parse_rss_items(xml_text):
    root = ET.fromstring(xml_text)
    items = []
    for item_el in root.findall(".//item"):
        title = (item_el.findtext("title") or "").strip()
        link = (item_el.findtext("link") or "").strip()
        desc_html = item_el.findtext("description") or ""
        pdf_url = None
        marker = 'class="download-file" href="'
        idx = desc_html.find(marker)
        if idx != -1:
            start = idx + len(marker)
            end = desc_html.find('"', start)
            pdf_url = desc_html[start:end]
        items.append({"titulo": title, "item_url": link, "pdf_url": pdf_url})
    return items

# dry-run: página 1 de "soja"
xml_text = fetch_rss_page("soja", page=1)
items = parse_rss_items(xml_text)
print(f"{len(items)} items encontrados")
for it in items[:10]:
    print(f"- {it['titulo']}  ->  {it['pdf_url']}")

10 items encontrados
- Resolución ex-SENASA N° 1039/1992  ->  https://biblioteca.senasa.gob.ar/files/original/f9e7bfec99154c15fe7bbd9b8548896a.pdf
- Resolución ex-SENASA N° 0918/1992  ->  https://biblioteca.senasa.gob.ar/files/original/cb0e2c20e567360d2f04592acd7613ad.PDF
- Resolución ex-SENASA N° 0399/1992  ->  https://biblioteca.senasa.gob.ar/files/original/1bc5fc4373d47ee8f5abe514e9ef7e45.pdf
- Resolución ex-SENASA N° 0990/1992  ->  https://biblioteca.senasa.gob.ar/files/original/2c04b81250f768806e301d6db2e59644.PDF
- Resolución ex-SENASA N° 0161/1992  ->  https://biblioteca.senasa.gob.ar/files/original/19e4d6f3ace6289f8e0c35d5619bfb4a.PDF
- Resolución SENASA N° 0402/2011  ->  https://biblioteca.senasa.gob.ar/files/original/5bb57ac807aac69eca7a0a53c310dad0.pdf
- Resolución SENASA N° 0296/2011 1° Período  ->  https://biblioteca.senasa.gob.ar/files/original/7f3550f277fa299adc5d2e80866142e3.pdf
- Resolución SENASA N° 0269/2011  ->  https://biblioteca.senasa.gob.ar/files/original/fe8082

CELDA 2

In [2]:
import time, json

DEFAULT_KEYWORDS = ["soja", "maiz", "trigo", "girasol", "oleaginosas", "granos", "cereales", "fitosanitario"]
RATE_LIMIT_SECONDS = 1.0

def search_keyword(keyword, max_pages=50):
    results = []
    page = 1
    while page <= max_pages:
        xml_text = fetch_rss_page(keyword, page)
        items = parse_rss_items(xml_text)
        if not items:
            break
        results.extend(items)
        print(f"  página {page}: {len(items)} items")
        if len(items) < 10:
            break  # última página
        page += 1
        time.sleep(RATE_LIMIT_SECONDS)
    return results

seen_urls = set()
all_items = []

for kw in DEFAULT_KEYWORDS:
    print(f"[buscando] '{kw}'")
    raw_items = search_keyword(kw)
    print(f"  -> {len(raw_items)} resultados totales para '{kw}'")
    for item in raw_items:
        if item["item_url"] not in seen_urls:
            seen_urls.add(item["item_url"])
            item["keyword_match"] = kw
            all_items.append(item)
    time.sleep(RATE_LIMIT_SECONDS)

print(f"\nTotal documentos únicos: {len(all_items)}")

with open("senasa_repositorio_index.json", "w", encoding="utf-8") as f:
    json.dump(all_items, f, ensure_ascii=False, indent=2)

# Descargar el archivo a tu computadora
from google.colab import files
files.download("senasa_repositorio_index.json")

[buscando] 'soja'
  página 1: 10 items
  página 2: 10 items
  página 3: 5 items
  -> 25 resultados totales para 'soja'
[buscando] 'maiz'
  página 1: 10 items
  página 2: 6 items
  -> 16 resultados totales para 'maiz'
[buscando] 'trigo'
  página 1: 10 items
  página 2: 7 items
  -> 17 resultados totales para 'trigo'
[buscando] 'girasol'
  página 1: 3 items
  -> 3 resultados totales para 'girasol'
[buscando] 'oleaginosas'
  página 1: 4 items
  -> 4 resultados totales para 'oleaginosas'
[buscando] 'granos'
  página 1: 10 items
  página 2: 10 items
  página 3: 10 items
  página 4: 10 items
  página 5: 10 items
  página 6: 3 items
  -> 53 resultados totales para 'granos'
[buscando] 'cereales'
  página 1: 10 items
  página 2: 10 items
  página 3: 2 items
  -> 22 resultados totales para 'cereales'
[buscando] 'fitosanitario'
  página 1: 10 items
  página 2: 10 items
  página 3: 10 items
  página 4: 10 items
  página 5: 10 items
  página 6: 10 items
  página 7: 10 items
  página 8: 10 items
  p

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

celda3

In [3]:
# Subí de nuevo el senasa_repositorio_index.json si reiniciaste el runtime
from google.colab import files
uploaded = files.upload()

import json
with open("senasa_repositorio_index.json", encoding="utf-8") as f:
    all_items = json.load(f)
print(f"{len(all_items)} documentos cargados")


Saving senasa_repositorio_index.json to senasa_repositorio_index (1).json
234 documentos cargados


celda 4

In [4]:
!pip install pypdf -q

import urllib.request, os, re, time
from pypdf import PdfReader

os.makedirs("pdfs", exist_ok=True)

CROP_PATTERNS = {
    "soja": [r"\bsoja\b", r"\bsoya\b"],
    "maiz": [r"\bma[ií]z\b"],
    "trigo": [r"\btrigo\b"],
    "girasol": [r"\bgirasol\b"],
    "oleaginosas": [r"\boleaginosa"],
    "granos": [r"\bgranos?\b", r"\bcereales?\b"],
}

def matched_crops(text):
    text_lower = text.lower()
    return [c for c, pats in CROP_PATTERNS.items() if any(re.search(p, text_lower) for p in pats)]

results = []

for i, item in enumerate(all_items):
    pdf_url = item.get("pdf_url")
    if not pdf_url:
        continue
    fname = f"pdfs/{i:04d}.pdf"
    try:
        req = urllib.request.Request(pdf_url, headers={"User-Agent": "agroexport-rag/0.1"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = resp.read()
        with open(fname, "wb") as f:
            f.write(data)
    except Exception as e:
        print(f"[error descarga] {item.get('titulo')}: {e}")
        continue
    time.sleep(0.5)

    try:
        reader = PdfReader(fname)
        text = "\n".join((p.extract_text() or "") for p in reader.pages)
    except Exception as e:
        print(f"[error extraccion] {item.get('titulo')}: {e}")
        text = ""

    crops = matched_crops(text)
    results.append({**item, "archivo_local": fname, "cultivos_mencionados": crops, "texto_extraido_chars": len(text)})

    if (i + 1) % 20 == 0:
        print(f"Procesados {i+1}/{len(all_items)}")

relevantes = [r for r in results if r["cultivos_mencionados"]]
print(f"\nTotal procesados: {len(results)}")
print(f"Con mención explícita de cultivo: {len(relevantes)}")
print(f"Sin mención (candidatos a descarte): {len(results) - len(relevantes)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 7.5 MB/s eta 0:00:00
Procesados 20/234


[error extraccion] Resolución SAGPyA N° 0006/2001: Stream has ended unexpectedly


[error extraccion] Resolución SAGPyA N° 0757 /1997: Stream has ended unexpectedly
Procesados 40/234
Procesados 60/234
Procesados 80/234
Procesados 100/234
Procesados 120/234
Procesados 140/234
Procesados 160/234
Procesados 180/234
Procesados 200/234


[error extraccion] Resolución SAGPyA N°830/2006: Stream has ended unexpectedly


[error extraccion] Resolución SAGPyA N° 1384/2004: Stream has ended unexpectedly
Procesados 220/234


[error extraccion] Resolución IASCAV N° 0409/1996: Stream has ended unexpectedly

Total procesados: 234
Con mención explícita de cultivo: 74
Sin mención (candidatos a descarte): 160


celda 5

In [5]:
# Guardar y descargar resultados
with open("senasa_index_con_texto.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
files.download("senasa_index_con_texto.json")

# Zippear solo los PDFs relevantes (los que sí mencionan algún cultivo)
import shutil
os.makedirs("pdfs_relevantes", exist_ok=True)
for r in relevantes:
    shutil.copy(r["archivo_local"], "pdfs_relevantes/")
shutil.make_archive("pdfs_relevantes", "zip", "pdfs_relevantes")
files.download("pdfs_relevantes.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>